In [29]:
from scipy.integrate import odeint 
import numpy as np
from matplotlib import pyplot as plt
from ipywidgets import interact, FloatSlider


def Model_Pseudomonas1(states, t, mu_max, k_subs, KLa):
    
    subs, biomass, Ca, Cl, Co, Cu, Fe, Mg, MoO4, Na, Zn,\
    K, Ni, NH4, P, S, CO2l, CO2g, O2l, O2g, H, OH = states


    '''
    Bioreactor/Physical constans
    '''
    R = 8.31451 # Rydberg's constant (KJ/mol/k)
    Tsc = 273.15 # Temperature scale constant (K)
    Tcelsius = 30 # Temperature (C)
    Tkelvin = Tcelsius + Tsc # Temperature (K)
    Pr = 101325 # Pressure (Pa)    


    '''
    Yield Factors:
    
    '''
    YCa = 1 #In this media we do not have Calcium, that is why we put 1
    YCl = 9.16
    YCo = 8.773
    YCu = 7.73
    YFe = 14.46
    YMg = 10.633
    YMoO4 = 7.773
    YNa = 7.467
    YZn = 0.00000002773
    YK = 0.156
    YNi = 0.00000006773
    YNH4 = 0.006046
    YP = 0.01193
    YS = 0.0003809
    YH = 0.09618
    YO2 = 18.74
    YCO2 = 20.43
    Ysubs = 1


    #Volumes

    Vf = 0
    Vs = 0
    Vl = 0.25
    Vgas = 0.05
    Vg = 0.105
    Voffgas = Vs + Vf + Vg
    Va = 0
    Vb = 0


    #Concentrations in feed
    Csubs_f = 0.000166/Vl
    CCa_f = 0/Vl
    CCo_f = 8.405819516967988e-07/Vl
    CCl_f = 0.000173/Vl
    CCu_f = 5.865708131748498e-08/Vl
    CFe_f = 1.798470580618242e-06/Vl
    CMg_f = 1.622889229695627e-04/Vl
    CMoO4_f = 1.239887682707782e-07/Vl
    CNa_f = 0.009841767287091/Vl
    CZn_f = 3.477547216397331e-07/Vl
    CK_f = 0.004408992875802/Vl
    CNi_f = 8.414299934452604e-08/Vl
    #CHCO3_f = 0.0012/Vl
    CP_f = 0.00863557426884131/Vl
    CS_f = 0.00319746255924881/Vl
    CNH4_f = 0.006054221608728/Vl
    CH_f = 0.0144375593208705/Vl
    COH_f = 0/Vl


    #Acid and base

    CH_a = 8.64
    COH_b = 4
    CNa_b = 4

    '''
    Gases
    '''
    Fraction_O2 = 0.21
    Fraction_CO2 = 0.000407

    TotalMolesGas = (P*Vgas)/(R * Tkelvin) #Total moles of gas
    PartialPressureO2 = Fraction_O2 * Pr #Partial pressure Oxygen
    PartialPressureCO2 = Fraction_CO2 * Pr #Partial pressure CO2

    HenryConstant_O2 = 0.000000013 #Henry's constant O2 (mol/L Pa)
    HenryConstant_CO2 = 0.00000034 #Henry's constant CO2 (mol/L Pa)

    CSat_O2 = PartialPressureO2 * HenryConstant_O2 #Saturation concentration O2 (mol/L)
    CSat_CO2 = PartialPressureCO2 * HenryConstant_CO2 #Saturation concentration CO2 (mol/L)

    Concentration_O2_l_initial = CSat_O2 #[O2] in liquid is equal to saturation concentration at the beggining
    Concentration_CO2_l_initial = CSat_CO2 #[CO2] in liquid is equal to saturation concentration at the beggining


    IC_O2_l = Concentration_O2_l_initial * Vl #Initial moles O2 in liquid
    IC_CO2_l = Concentration_CO2_l_initial * Vl #Initial moles CO2 in liquid 

    IC_CO2_g = Fraction_CO2 * TotalMolesGas #Initial moles of CO2 in gas
    IC_O2_g = Fraction_O2 * TotalMolesGas #Initial moles of O2 in gas

    CO2_g_in = IC_CO2_g/Vgas #Concentration in the purge
    O2_g_in = IC_O2_g/Vgas #Concentration in the purge

    mu = mu_max * (subs/(k_subs + subs))

    dsubs_dt = Csubs_f * Vf - (subs/Vl) * Vs - 1/Ysubs * mu * biomass

    dbiomass_dt = mu*biomass - (biomass/Vl) * Vs 

    dCa_dt = CCa_f * Vf - (Ca/Vl) * Vs - 1/YCa * mu * biomass

    dCl_dt = CCl_f * Vf - (Cl/Vl) * Vs - 1/YCl * mu * biomass

    dCo_dt = CCo_f * Vf - (Co/Vl) * Vs - 1/YCo * mu * biomass

    dCu_dt = CCu_f * Vf - (Cu/Vl) * Vs - 1/YCu * mu * biomass

    dFe_dt = CFe_f * Vf - (Fe/Vl) * Vs - 1/YFe * mu * biomass

    dMg_dt = CMg_f * Vf - (Mg/Vl)  * Vs - 1/YMg * mu * biomass

    dMoO4_dt = CMoO4_f * Vf - (MoO4/Vl) * Vs - 1/YMoO4 * mu * biomass

    dNa_dt = CNa_f * Vf + CNa_b * Vb - (Na/Vl) * Vs - 1/YNa * mu * biomass

    dZn_dt = CZn_f * Vf - (Zn/Vl) * Vs - 1/YZn * mu * biomass

    dK_dt = CK_f * Vf - (K/Vl) * Vs - 1/YK * mu * biomass

    dNi_dt = CNi_f * Vf - (Ni/Vl) * Vs - 1/YNi * mu * biomass

    dNH4_dt = CNH4_f * Vf - (NH4/Vl) * Vs - 1/YNH4 * mu * biomass

    dP_dt = CP_f * Vf - (P/Vl) * Vs - 1/YP * mu * biomass

    dS_dt = CS_f * Vf - (S/Vl) * Vs - 1/YS * mu * biomass

    dCO2l_dt = KLa * (CSat_CO2 - (CO2l/Vl)) * Vl  + 1/YCO2 * mu * biomass

    dCO2g_dt = CO2_g_in * Vg - (KLa * (CSat_CO2 - (CO2l/Vl)) * Vl) - (CO2g/Vgas)*Voffgas

    dO2l_dt = KLa * (CSat_O2 - O2l) * Vl - 1/YO2*mu* biomass

    dO2g_dt = O2_g_in * Vg - (KLa * (CSat_O2 - O2l) * Vl ) - (O2g/Vgas)*Voffgas

    dH_dt = CH_f * Vf + CH_a * Va - (H/Vl) * Vs + 1/YH * mu * biomass

    dOH_dt = COH_b*Vb + COH_f * Vf - (OH/Vl)* Vs


    return dsubs_dt, dbiomass_dt, dCa_dt, dCl_dt, dCo_dt, dCu_dt, dFe_dt, \
           dMg_dt, dMoO4_dt, dNa_dt, dZn_dt, dK_dt, dNi_dt, dNH4_dt, dP_dt,\
           dS_dt, dCO2l_dt, dCO2g_dt, dO2l_dt, dO2g_dt, dH_dt, dOH_dt

    

In [30]:
def run(mu_max=0.013, k_subs=25, KLa=2):
    
    # Initial conditions
    subs = 0.13
    biomass = 0.04267
    Ca = 0
    Cl = 0.000173
    Co = 8.405819516967988e-07
    Cu = 5.865708131748498e-08
    Fe = 1.798470580618242e-06
    Mg = 1.622889229695627e-04
    MoO4 = 1.239887682707782e-07
    Na = 0.009841767287091
    Zn = 3.477547216397331e-07
    K = 0.004408992875802
    Ni = 8.414299934452604e-08
    NH4 = 0.006054221608728
    P = 0.00863557426884131
    S = 0.00319746255924881
    H = 0.0144375593208705
    OH = 0

    '''
    For initial conditions of gases
    '''
    KLa = 2
    R = 8.31451 # Rydberg's constant (KJ/mol/k)
    Tsc = 273.15 # Temperature scale constant (K)
    Tcelsius = 30 # Temperature (C)
    Tkelvin = Tcelsius + Tsc # Temperature (K)
    Pr = 101325 # Pressure (Pa)
    Ro = 1000 # Density of the cultivation medium (g/L)
    Vgas = 0.05
    Fraction_O2 = 0.21
    Fraction_CO2 = 0.000407
    Vl = 0.25

    TotalMolesGas = (Pr*Vgas)/(R * Tkelvin) #Total moles of gas
    PartialPressureO2 = Fraction_O2 * Pr #Partial pressure Oxygen
    PartialPressureCO2 = Fraction_CO2 * Pr #%Partial pressure CO2

    HenryConstant_O2 = 0.000000013 #Henry's constant O2 (mol/L Pa)
    HenryConstant_CO2 = 0.00000034 #Henry's constant CO2 (mol/L Pa)

    CSat_O2 = PartialPressureO2 * HenryConstant_O2 #Saturation concentration O2 (mol/L)
    CSat_CO2 = PartialPressureCO2 * HenryConstant_CO2 #Saturation concentration CO2 (mol/L)

    Concentration_O2_l_initial = CSat_O2 #[O2] in liquid is equal to saturation concentration at the beggining
    Concentration_CO2_l_initial = CSat_CO2 #[CO2] in liquid is equal to saturation concentration at the beggining

    Initial_OTR = KLa *(CSat_O2 - Concentration_O2_l_initial) #Initial Oxygen Transfer Rate (mol/L*min)
    Initial_CO2TR = KLa * (CSat_CO2 - Concentration_CO2_l_initial) #Initial CO2 Transfer Rate (mol/L*min)

    IC_O2_l = Concentration_O2_l_initial * Vl #Initial moles O2 in liquid
    IC_CO2_l = Concentration_CO2_l_initial * Vl #Initial moles CO2 in liquid 

    IC_CO2_g = Fraction_CO2 * TotalMolesGas #Initial moles of CO2 in gas
    IC_O2_g = Fraction_O2 * TotalMolesGas #Initial moles of O2 in gas

    CO2_g_in = IC_CO2_g/Vgas #Concentration in the purge
    O2_g_in = IC_O2_g/Vgas #Concentration in the purge

    CO2l = IC_CO2_l
    CO2g = IC_CO2_g
    O2l = IC_O2_l
    O2g = IC_O2_g

   
    IC = [subs, biomass, Ca, Cl, Co, Cu, Fe, Mg, MoO4, Na, Zn,\
          K, Ni, NH4, P, S, CO2l, CO2g, O2l, O2g, H, OH]

    t = np.linspace(0,8000,1500)
    X = odeint(Model_Pseudomonas1, IC, t, args=(mu_max, k_subs, KLa))
    
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,0], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('Substrate')

    axs[0, 1].plot(t, X[:,1], color='crimson')
    axs[0, 1].set_xlabel('Time (minutes)')
    axs[0, 1].set_ylabel('Biomass') 

    axs[1, 0].plot(t, X[:,2], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('Calcium')

    axs[1, 1].plot(t, X[:,3], color='crimson')
    axs[1, 1].set_xlabel('Time (minutes)')
    axs[1, 1].set_ylabel('Chlorine')
    
    plt.tight_layout()
    

    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,4], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('Cobaltium')

    axs[0, 1].plot(t, X[:,5], color='crimson')
    axs[0, 1].set_xlabel('Time (minutes)')
    axs[0, 1].set_ylabel('Cupper') 

    axs[1, 0].plot(t, X[:,6], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('Iron')

    axs[1, 1].plot(t, X[:,7], color='crimson')
    axs[1, 1].set_xlabel('Time (minutes)')
    axs[1, 1].set_ylabel('Magnesium')

    plt.tight_layout()


    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,8], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('Molybdenum')

    axs[0, 1].plot(t, X[:,9], color='crimson')
    axs[0, 1].set_xlabel('Time (minutes)')
    axs[0, 1].set_ylabel('Sodium') 

    axs[1, 0].plot(t, X[:,10], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('Zinc')

    axs[1, 1].plot(t, X[:,11], color='crimson')
    axs[1, 1].set_xlabel('Time (minutes)')
    axs[1, 1].set_ylabel('Potassium')

    plt.tight_layout()
    
    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,12], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('Niquel')

    axs[0, 1].plot(t, X[:,13], color='crimson')
    axs[0, 1].set_xlabel('Time (minutes)')
    axs[0, 1].set_ylabel('Ammonia') 

    axs[1, 0].plot(t, X[:,14], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('Phosphorus')

    axs[1, 1].plot(t, X[:,15], color='crimson')
    axs[1, 1].set_xlabel('Time (minutes)')
    axs[1, 1].set_ylabel('Sulfate')
    
    plt.tight_layout()

    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,16], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('CO2 liquid')

    axs[0, 1].plot(t, X[:,17], color='crimson')
    axs[0, 1].set_xlabel('Time (minutes)')
    axs[0, 1].set_ylabel('CO2 gas') 

    axs[1, 0].plot(t, X[:,18], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('O2 liquid')

    axs[1, 1].plot(t, X[:,19], color='crimson')
    axs[1, 1].set_xlabel('Time (minutes)')
    axs[1, 1].set_ylabel('O2 gas')
    
    plt.tight_layout()

    fig, axs = plt.subplots(2, 2, figsize=(10, 10))

    axs[0, 0].plot(t, X[:,20], color='crimson')
    axs[0, 0].set_xlabel('Time (minutes)')
    axs[0, 0].set_ylabel('Protons')

    axs[1, 0].plot(t, X[:,21], color='crimson')
    axs[1, 0].set_xlabel('Time (minutes)')
    axs[1, 0].set_ylabel('OH')

    plt.tight_layout()
    plt.show()






interact(run,
         mu_max=FloatSlider(value=0.013, min=0.001, max=0.05, step=0.001, description='μmax'),
         k_subs=FloatSlider(value=25, min=0.1, max=100, step=1, description='Ks'),
         KLa=FloatSlider(value=2, min=0.1, max=10, step=0.1, description='KLa'))

interactive(children=(FloatSlider(value=0.013, description='μmax', max=0.05, min=0.001, step=0.001), FloatSlid…

<function __main__.run(mu_max=0.013, k_subs=25, KLa=2)>